# 0 . Library

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import biosppy

# 1 . Raw Data to PreProcessed
Adding name of features to columns

In [ ]:
Data_path = '/home/vault/empkins/tpD/D02/Students/Yasaman/BioSig_data/BioSig_RCT_Data_Info/RCT.csv'

Shivam_data = [
    '005', '171', '239', '318', '430', '467', '482', '538', '566', '576', '611', '629', '663', '666', '699',
    '720', '732', '746', '760', '777', '792', '808', '846', '857', '861', '873', '887', '910',
    '040', '199', '242', '385', '444', '471', '498', '540', '567', '597', '615', '655', '664', '673', '705',
    '726', '735', '747', '763', '783', '795', '824', '851', '858', '865', '875', '906',
    '074', '207', '295', '389', '453', '476', '511', '552', '569', '606', '625', '662', '665', '692', '709',
    '729', '741', '753', '770', '788', '806', '834', '856', '859', '866', '877', '907'
]
Shivam_data = [int(x) for x in X]

df_RCT = pd.read_csv(Data_path)
df_RCT.head()

RCT_data = list(df_base['ID'])  # <-- your list of IDs
IDs = [x for x in RCT_data if x not in Shivam_data]




# Base paths
raw_data_base = '/home/vault/empkins/tpD/D02/RCT/raw_data'
processed_data_base = '/home/vault/empkins/tpD/D02/Students/Yasaman/BioSig_data'

for id_num in IDs:
    participant_id = str(id_num).zfill(3)  # Pad ID to 3 digits

    participant_folder = os.path.join(raw_data_base, participant_id)
    #base_save_folder = os.path.join(processed_data_base, participant_id)

    if not os.path.exists(participant_folder):
        print(f"Warning: Folder for ID {participant_id} does not exist. Skipping.\n")
        continue

    # Find all .txt files
    txt_files = [f for f in os.listdir(participant_folder) if f.endswith('.txt')]

    if not txt_files:
        print(f"Warning: No .txt files found for ID {participant_id}. Skipping.\n")
        continue

    # Prefer normal .txt (without _ts)
    normal_txt_files = [f for f in txt_files if '_ts' not in f]

    if normal_txt_files:
        chosen_file = normal_txt_files[0]
    else:
        chosen_file = txt_files[0]

    raw_file_path = os.path.join(participant_folder, chosen_file)
    print(f"Processing {chosen_file} for ID {participant_id}...")

    try:
        # Load the raw data
        df = pd.read_csv(raw_file_path, skiprows=21, sep="\t", header=None)

        # Drop unimportant columns
        df.drop(columns=[0, 1, 8], inplace=True)

        # Rename columns
        df.columns = ['RSP', 'Corrugator', 'Oculi', 'Zygomaticus', 'Masseter', 'ECG']

        # ---- Save into /processed_data/{ID}/{ID}/PreProcessed.csv ----
        save_folder = os.path.join(processed_data_base, participant_id)
        os.makedirs(save_folder, exist_ok=True)  # Create subfolder

        save_file_path = os.path.join(save_folder, 'PreProcessed.csv')

        df.to_csv(save_file_path, index=False)
        print(f"Saved preprocessed file for ID {participant_id} at {save_file_path}.\n")

    except Exception as e:
        print(f"Error processing ID {participant_id}: {str(e)}\n")


# 2 . PreProcessed to Data
Extracting specific parts of data based on Journal start time and labels from App file.

In [ ]:
# Base paths
preprocessed_base = '/home/vault/empkins/tpD/D02/Students/Yasaman/BioSig_data'  # <-- Where you want to save
metadata_base = '/home/vault/empkins/tpD/D02/RCT/raw_data'  # <-- Where the metadata and raw data are

for id_num in IDs:
    participant_id = str(id_num).zfill(3)

    PreProcessed_PATH = os.path.join(preprocessed_base, participant_id)
    METADATA_PATH = os.path.join(metadata_base, participant_id)

    print(f"\nProcessing ID {participant_id}...")

    if not os.path.exists(PreProcessed_PATH):
        print(f"Warning: PreProcessed_PATH for ID {participant_id} does not exist. Skipping.\n")
        continue

    if not os.path.exists(METADATA_PATH):
        print(f"Warning: METADATA_PATH for ID {participant_id} does not exist. Skipping.\n")
        continue

    # Get available files
    PreProcessed_FILE = os.listdir(PreProcessed_PATH)
    print(f'Preprocessed Files: {PreProcessed_FILE}')

    META_FILES = os.listdir(METADATA_PATH)
    print(f'Metadata Files: {META_FILES}')

    # --- 1. Find the metadata .xls file ---

    excel_files = [f for f in META_FILES if f.endswith('.xls')]
    if not excel_files:
        print(f"Warning: No metadata .xls file found for ID {participant_id}. Skipping.\n")
        continue
    meta_file_path = os.path.join(METADATA_PATH, excel_files[0])

    # Load Start and Marker Info
    start_and_marker_info = pd.read_excel(meta_file_path, engine='xlrd', header=1)

    print(f"Start & Marker file loaded for ID {participant_id}.")

    #test_start_timestamp = start_and_marker_info.loc[0, "Date Created"]
    test_start_timestamp = start_and_marker_info.loc[0, "Date Created"]
    print(f'Test Start Timestamp: {type(test_start_timestamp)}')
    print(f'Test Start Timestamp: {test_start_timestamp}')

    # --- 2. Find the _app.csv file smartly ---
    correct_app_folder = os.path.join(METADATA_PATH, 'correct_app_labels')

    if os.path.exists(correct_app_folder):
        # Prefer _app.csv inside correct_app_labels folder
        app_files = [f for f in os.listdir(correct_app_folder) if f.endswith('app.csv')]
        if app_files:
            phases_file_path = os.path.join(correct_app_folder, app_files[0])
            print(f"Found _app.csv in correct_app_labels for ID {participant_id}.")
        else:
            print(f"Warning: No _app.csv found inside correct_app_labels for ID {participant_id}. Skipping.\n")
            continue
    else:
        # Otherwise, fallback to metadata folder
        app_files = [f for f in os.listdir(METADATA_PATH) if f.endswith('.csv')]
        if app_files:
            phases_file_path = os.path.join(METADATA_PATH, app_files[0])
            print(f"Found _app.csv in metadata folder for ID {participant_id}.")
        else:
            print(f"Warning: No _app.csv found for ID {participant_id}. Skipping.\n")
            continue

    # Read the phases data
    phases_data = pd.read_csv(phases_file_path)
    phases_data['timestamp'] = phases_data['timestamp'].str.replace('\n', '')
    print('Timestamp & Labels:')
    print(phases_data.head(10))

    # Saving 1
    Save_folder = os.path.join(preprocessed_base, participant_id)
    save_file_path = os.path.join(Save_folder, 'label_timestamp.csv')
    phases_data.to_csv(save_file_path, index=False)

    # --- 3. Calculate start index ---
    actual_test_start = phases_data.loc[phases_data['label'] == "lat_instr_01", "timestamp"]
    if actual_test_start.empty:
        print(f"Warning: 'lat_instr_01' not found in phases for ID {participant_id}. Skipping.\n")
        continue

    print(f'Actual Test Start: {actual_test_start.values}')

    # Compute how many 0.5 ms intervals have passed
    diff = pd.to_datetime(actual_test_start) - pd.to_datetime(test_start_timestamp)
    start_index = (diff.dt.total_seconds() * 1000 // 0.5).astype(int)
    start_index = int(start_index.iloc[0])
    print(f'Start index: {start_index}')


    # --- 4. Load PreProcessed Data ---
    preprocessed_file_path = os.path.join(PreProcessed_PATH, 'PreProcessed.csv')
    if not os.path.exists(preprocessed_file_path):
        print(f"Warning: PreProcessed.csv not found for ID {participant_id}. Skipping.\n")
        continue

    # Saving 2
    df = pd.read_csv(preprocessed_file_path)
    print('Shape Before preprocessing:', df.shape)

    df_new = df[start_index:]
    print('Shape After preprocessing:', df_new.shape)

    #Saving 3
    # --- 5. Save Final Data ---
    save_file_path = os.path.join(Save_folder, 'Data.csv')
    df_new.to_csv(save_file_path, index=False)
    print(f"Data.csv saved for ID {participant_id} at {save_file_path}.\n")


# 3 . SNR Calculation
Calculating SNR for ECG Datasets

In [ ]:
########### SNR_Calculation_USING_BIOSPPY
def calculate_snr(cleaned_signal, raw_signal, verbose=False):
    cleaned_signal.ffill(inplace=True)
    cleaned_signal.bfill(inplace=True)
    raw_signal.ffill(inplace=True)
    raw_signal.bfill(inplace=True)

    if len(cleaned_signal) == 0:
        return float('nan')

    noise = raw_signal - cleaned_signal
    signal_power = np.mean(cleaned_signal**2)
    noise_power = np.mean(noise**2)

    if noise_power == 0:
        if verbose:
            print("Warning: Noise power is zero")
        return float('nan')
    snr_db = 10 * np.log10(signal_power / noise_power if noise_power else 1)
    return snr_db



SNR_results_df = pd.DataFrame(columns=['ID', 'SNR'])

for ID in IDs:
    print(f"Start processing: {ID}")
    ID = str(ID).zfill(3)
    raw_signal_path = f"/home/vault/empkins/tpD/D02/Students/Yasaman/BioSig_data/{ID}/Data.csv"
    try:
        raw_signal = pd.read_csv(raw_signal_path).ECG
        # Step 3: Safety check
        if raw_signal.isna().all() or len(raw_signal) < 1000:
            print(f"Signal too short or contains only NaNs for ID {ID}")
            continue
        cleaned_ecg = biosppy.signals.ecg.ecg(signal=raw_signal, sampling_rate=2000, show=False)
        cleaned_ecg = pd.DataFrame(data=cleaned_ecg['filtered'], columns=['ECG'])['ECG']
    except ValueError:
        print("Value Error occured!")
        continue
    except FileNotFoundError:
        print(f"Files for ID {ID} not found.")
        continue
    except pd.errors.EmptyDataError:
        print(f"Data for ID {ID} is empty or corrupted.")
        continue
    snr = calculate_snr(cleaned_ecg, raw_signal, verbose=True)
    if pd.isna(snr):
        print(f"SNR calculation failed or resulted in NaN for ID {ID}.")
        continue
    temp_df = pd.DataFrame({'ID': [ID], 'SNR': [snr]})
    SNR_results_df = pd.concat([SNR_results_df, temp_df], ignore_index=True)
    print(f"ID: {ID} Processed..\n")

SNR_results_df.to_csv("/home/vault/empkins/tpD/D02/Students/Yasaman/BioSig_data/Processed Data/ECG_SNR_BIOSPPY.csv", index=False)

# 4 . Apply Windowing function

In [ ]:
def apply_windowing(df, window_size=1000):
    # Belongs to Utilities
    """
    Splits a DataFrame into non-overlapping windows of a specified size.

    Parameters:
        df (pd.DataFrame): The input DataFrame to be split into windows.
        window_size (int): The number of rows in each window. Default is 1000.

    Returns:
        list: A list of DataFrames, where each DataFrame represents a window
              containing `window_size` rows. If the number of rows in `df` is
              not perfectly divisible by `window_size`, the last window will
              contain the remaining rows.
    """
    num_windows = len(df) // window_size
    windows = []

    for i in range(num_windows):
        window_start = i * window_size
        window_end = (i + 1) * window_size
        window_data = df.iloc[window_start:window_end]
        windows.append(window_data)

    if len(df) % window_size != 0:
        last_window = df.iloc[num_windows * window_size:]
        windows.append(last_window)

    return windows

# 5 . RSP : Filtering + Windowing + Feature Extraction

In [ ]:
import pandas as pd
import neurokit2 as nk
import os
import pandas as pd
import glob
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
import numpy as np
import sys

In [ ]:
SAMPLING_RATE = 2000
Window_Time = 10 # 10', 5', 4', 3'
WINDOW_SIZE = Window_Time * 60 * 2000

In [ ]:
def apply_windowing(df, window_size=1000):
    # Belongs to Utilities
    """
    Splits a DataFrame into non-overlapping windows of a specified size.

    Parameters:
        df (pd.DataFrame): The input DataFrame to be split into windows.
        window_size (int): The number of rows in each window. Default is 1000.

    Returns:
        list: A list of DataFrames, where each DataFrame represents a window
              containing `window_size` rows. If the number of rows in `df` is
              not perfectly divisible by `window_size`, the last window will
              contain the remaining rows.
    """
    num_windows = len(df) // window_size
    windows = []

    for i in range(num_windows):
        window_start = i * window_size
        window_end = (i + 1) * window_size
        window_data = df.iloc[window_start:window_end]
        windows.append(window_data)

    if len(df) % window_size != 0:
        last_window = df.iloc[num_windows * window_size:]
        windows.append(last_window)

    return windows

def extract_features(signal, window_size, overlap, ID):
    features = []
    step_size = int(window_size * (1 - overlap))
    for start in range(0, len(signal) - window_size, step_size):
        window = signal[start:start + window_size]
        window_cleaned = nk.rsp_clean(window, sampling_rate=2000)
        df, peaks_dict = nk.rsp_peaks(window_cleaned)
        info = nk.rsp_fixpeaks(peaks_dict)
        rsp_rate = nk.rsp_rate(window_cleaned, peaks_dict, sampling_rate=2000)
        rrv_features = nk.rsp_rrv(rsp_rate, info, sampling_rate=2000, show=False)
        rrv_features["mean"] = np.mean(window)
        rrv_features["std"] = np.std(window)
        rrv_features["min"] = np.min(window)
        rrv_features["max"] = np.max(window)
        rrv_features["median"] = np.median(window)
        rrv_features["ID"] = ID
        features.append(rrv_features)
    return features

In [ ]:
Dataframes = []
overlap = 0
cnt = 1
for ID in IDs:
    print(f"{cnt}. Start Processing {ID}")
    try:
        signal = pd.read_csv(f'/home/vault/empkins/tpD/D02/Students/Yasaman/BioSig_data/{ID}/Data.csv').RSP
        features = extract_features(signal, WINDOW_SIZE, overlap, ID)
        Dataframes.append(features)
        del features
    except FileNotFoundError:
        print(f"Files for ID {ID} not found.")
        continue
    print(f'Features Extracted for ID {ID}\n')
    cnt +=1

df_RSP = pd.concat([df for sublist in Dataframes for df in sublist]).reset_index(drop=True)
df_RSP.to_csv(f"/home/vault/empkins/tpD/D02/Students/Yasaman/BioSig_data/Processed Data/RSP/RSP_78_78_H_{Window_Time}_Minute.csv", index=False)

# 6 . ECG: Filtering + Windowing + Feature Extraction

In [ ]:
import pandas as pd
import neurokit2 as nk
import sys
import os
import pandas as pd
import glob
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
SAMPLING_RATE = 2000
Window_Time = 3 # 10, 5, 4, 3, 2, 1
WINDOW_SIZE = Window_Time * 60 * 2000

In [ ]:
def apply_windowing(df, window_size=1000):
    # Belongs to Utilities
    """
    Splits a DataFrame into non-overlapping windows of a specified size.

    Parameters:
        df (pd.DataFrame): The input DataFrame to be split into windows.
        window_size (int): The number of rows in each window. Default is 1000.

    Returns:
        list: A list of DataFrames, where each DataFrame represents a window
              containing `window_size` rows. If the number of rows in `df` is
              not perfectly divisible by `window_size`, the last window will
              contain the remaining rows.
    """
    num_windows = len(df) // window_size
    windows = []

    for i in range(num_windows):
        window_start = i * window_size
        window_end = (i + 1) * window_size
        window_data = df.iloc[window_start:window_end]
        windows.append(window_data)

    if len(df) % window_size != 0:
        last_window = df.iloc[num_windows * window_size:]
        windows.append(last_window)

    return windows



def extract_features(signal, window_size, overlap, ID):
    features = []
    step_size = int(window_size * (1 - overlap))
    for start in range(0, len(signal) - window_size, step_size):
        window = signal[start:start + window_size]
        window = nk.ecg_clean(window, sampling_rate=2000)
        peaks, info = nk.ecg_peaks(window, sampling_rate=2000, correct_artifacts=True)
        hrv = nk.hrv(peaks, sampling_rate=2000, show=False)
        hrv['ID'] = ID
        features.append(hrv)
    return features

In [ ]:
Dataframes = []
overlap = 0
cnt = 1
for ID in IDs:
    print(f'{cnt}. Start processing for ID {ID}')
    try:
        signal = pd.read_csv(f'/home/vault/empkins/tpD/D02/Students/Yasaman/BioSig_data/{ID}/Data.csv').ECG
        features = extract_features(signal, WINDOW_SIZE, overlap, ID)
        Dataframes.append(features)
        del features
    except FileNotFoundError:
        print(f"Files for ID {ID} not found.")
        continue
    cnt += 1
    print(f'Features Extracted for ID {ID}\n')

df_ECG = pd.concat([df for sublist in Dataframes for df in sublist]).reset_index(drop=True)
df_ECG.to_csv(f"/home/vault/empkins/tpD/D02/Students/Yasaman/BioSig_data/Processed Data/ECG/ECG_75_75_H_{Window_Time}_Minute.csv", index=False)

# 7 . EMG Including Masseter

In [ ]:
import os
import pandas as pd
import sys
import neurokit2 as nk
import numpy as np
from biosppy.signals import emg
from scipy.signal import welch

In [ ]:
SAMPLING_RATE = 2000
Window_Time = 3 # 10', 5', 4', 3'
WINDOW_SIZE = Window_Time * 60 * 2000

In [ ]:
def calculate_rmse(emg_signal, sampling_rate):
    ts, filtered, _ = emg.emg(signal=emg_signal, sampling_rate=sampling_rate, show=False)
    return np.sqrt(np.mean(filtered**2))

def calculate_mav(emg_signal, sampling_rate):
    ts, filtered, _ = emg.emg(signal=emg_signal, sampling_rate=sampling_rate, show=False)
    return np.mean(np.abs(filtered))

def calculate_var(emg_signal, sampling_rate):
    ts, filtered, _ = emg.emg(signal=emg_signal, sampling_rate=sampling_rate, show=False)
    return np.var(filtered)

def calculate_energy(emg_signal, sampling_rate):
    ts, filtered, _ = emg.emg(signal=emg_signal, sampling_rate=sampling_rate, show=False)
    return np.sum(filtered**2)


def calculate_mnf_mdf_fr(emg_signal, sampling_rate):
    f, psd = welch(emg_signal, fs=sampling_rate, nperseg=512)

    mnf = np.sum(f * psd) / np.sum(psd)

    cumulative_power = np.cumsum(psd)
    total_power = np.sum(psd)
    mdf_index = np.where(cumulative_power >= total_power / 2)[0][0]
    mdf = f[mdf_index]

    fr_index = np.argmax(psd)
    fr = f[fr_index] / mnf

    return mnf, mdf, fr

def z_score_normalization(signal):
    mean = np.mean(signal)
    std_dev = np.std(signal)
    if std_dev == 0:
        std_dev = 1
    normalized_signal = (signal - mean) / std_dev
    return normalized_signal

def extract_emg_features(dataframe: pd.DataFrame, normalize=False):
    dataframe = dataframe.copy()

    if normalize:
        dataframe['Corrugator'] = z_score_normalization(dataframe['Corrugator'].values)
        dataframe['Oculi'] = z_score_normalization(dataframe['Oculi'].values)
        dataframe['Zygomaticus'] = z_score_normalization(dataframe['Zygomaticus'].values)
        dataframe['Masseter'] = z_score_normalization(dataframe['Masseter'].values)


    standard_features = dataframe.describe()
    c_mean_value = standard_features.at['mean', 'Corrugator']
    o_mean_value = standard_features.at['mean', 'Oculi']
    z_mean_value = standard_features.at['mean', 'Zygomaticus']
    m_mean_value = standard_features.at['mean', 'Masseter']

    c_std_value = standard_features.at['std', 'Corrugator']
    o_std_value = standard_features.at['std', 'Oculi']
    z_std_value = standard_features.at['std', 'Zygomaticus']
    m_std_value = standard_features.at['std', 'Masseter']

    c_min_value = standard_features.at['min', 'Corrugator']
    o_min_value = standard_features.at['min', 'Oculi']
    z_min_value = standard_features.at['min', 'Zygomaticus']
    m_min_value = standard_features.at['min', 'Masseter']

    c_max_value = standard_features.at['max', 'Corrugator']
    o_max_value = standard_features.at['max', 'Oculi']
    z_max_value = standard_features.at['max', 'Zygomaticus']
    m_max_value = standard_features.at['max', 'Masseter']

    c_range_value = c_max_value - c_min_value
    o_range_value = o_max_value - o_min_value
    z_range_value = z_max_value - z_min_value
    m_range_value = m_max_value - m_min_value



    c_rmse_value = calculate_rmse(dataframe['Corrugator'].values, SAMPLING_RATE)
    o_rmse_value = calculate_rmse(dataframe['Oculi'].values, SAMPLING_RATE)
    z_rmse_value = calculate_rmse(dataframe['Zygomaticus'].values, SAMPLING_RATE)
    m_rmse_value = calculate_rmse(dataframe['Masseter'].values, SAMPLING_RATE)


    c_mav_value = calculate_mav(dataframe['Corrugator'].values, SAMPLING_RATE)
    o_mav_value = calculate_mav(dataframe['Oculi'].values, SAMPLING_RATE)
    z_mav_value = calculate_mav(dataframe['Zygomaticus'].values, SAMPLING_RATE)
    m_mav_value = calculate_mav(dataframe['Masseter'].values, SAMPLING_RATE)


    c_var_value = calculate_var(dataframe['Corrugator'].values, SAMPLING_RATE)
    o_var_value = calculate_var(dataframe['Oculi'].values, SAMPLING_RATE)
    z_var_value = calculate_var(dataframe['Zygomaticus'].values, SAMPLING_RATE)
    m_var_value = calculate_var(dataframe['Masseter'].values, SAMPLING_RATE)

    c_energy_value = calculate_energy(dataframe['Corrugator'].values, SAMPLING_RATE)
    o_energy_value = calculate_energy(dataframe['Oculi'].values, SAMPLING_RATE)
    z_energy_value = calculate_energy(dataframe['Zygomaticus'].values, SAMPLING_RATE)
    m_energy_value = calculate_energy(dataframe['Masseter'].values, SAMPLING_RATE)

    c_mnf_value, c_mdf_value, c_fr_value = calculate_mnf_mdf_fr(dataframe['Corrugator'].values, SAMPLING_RATE)
    o_mnf_value, o_mdf_value, o_fr_value = calculate_mnf_mdf_fr(dataframe['Oculi'].values, SAMPLING_RATE)
    z_mnf_value, z_mdf_value, z_fr_value = calculate_mnf_mdf_fr(dataframe['Zygomaticus'].values, SAMPLING_RATE)
    m_mnf_value, m_mdf_value, m_fr_value = calculate_mnf_mdf_fr(dataframe['Masseter'].values, SAMPLING_RATE)

    dataframe_final = pd.DataFrame({
        'emg_c_mean': [c_mean_value],
        'emg_o_mean': [o_mean_value],
        'emg_z_mean': [z_mean_value],
        'emg_m_mean': [m_mean_value],
        'emg_c_std': [c_std_value],
        'emg_o_std': [o_std_value],
        'emg_z_std': [z_std_value],
        'emg_m_std': [m_std_value],
        'emg_c_min': [c_min_value],
        'emg_o_min': [o_min_value],
        'emg_z_min': [z_min_value],
        'emg_m_min': [m_min_value],
        'emg_c_max': [c_max_value],
        'emg_o_max': [o_max_value],
        'emg_z_max': [z_max_value],
        'emg_m_max': [m_max_value],
        'emg_c_range': [c_range_value],
        'emg_o_range': [o_range_value],
        'emg_z_range': [z_range_value],
        'emg_m_range': [m_range_value],
        'emg_c_rmse': [c_rmse_value],
        'emg_o_rmse': [o_rmse_value],
        'emg_z_rmse': [z_rmse_value],
        'emg_m_rmse': [m_rmse_value],
        'emg_c_mav': [c_mav_value],
        'emg_o_mav': [o_mav_value],
        'emg_z_mav': [z_mav_value],
        'emg_m_mav': [m_mav_value],
        'emg_c_var': [c_var_value],
        'emg_o_var': [o_var_value],
        'emg_z_var': [z_var_value],
        'emg_m_var': [m_var_value],
        'emg_c_energy': [c_energy_value],
        'emg_o_energy': [o_energy_value],
        'emg_z_energy': [z_energy_value],
        'emg_m_energy': [m_energy_value],
        'emg_c_mnf': [c_mnf_value],
        'emg_o_mnf': [o_mnf_value],
        'emg_z_mnf': [z_mnf_value],
        'emg_m_mnf': [m_mnf_value],
        'emg_c_mdf': [c_mdf_value],
        'emg_o_mdf': [o_mdf_value],
        'emg_z_mdf': [z_mdf_value],
        'emg_m_mdf': [m_mdf_value],
        'emg_c_fr': [c_fr_value],
        'emg_o_fr': [o_fr_value],
        'emg_z_fr': [z_fr_value],
        'emg_m_fr': [m_fr_value],
    })
    return dataframe_final

In [ ]:
Dataframes = []
cnt = 1
for ID in IDs:
    df = pd.read_csv(f'/home/vault/empkins/tpD/D02/Students/Yasaman/BioSig_data/{ID}/Data.csv')
    print(f"{cnt}. Loaded Signal for {ID}")

    emg_signals = [df['Corrugator'].values, df['Oculi'].values, df['Zygomaticus'].values, df['Masseter'].values]
    emg_cleaned = [nk.emg_clean(emg, sampling_rate=SAMPLING_RATE) for emg in emg_signals]
    cleaned_df = pd.DataFrame({
        'Corrugator': emg_cleaned[0],
        'Oculi': emg_cleaned[1],
        'Zygomaticus': emg_cleaned[2],
        'Masseter': emg_cleaned[3]
    })
    print(f"Cleaned Signal for {ID}")

    windows = apply_windowing(cleaned_df, WINDOW_SIZE)
    print(f"Applied Windowing for {ID}")
    print(f"Windows for {ID} is {len(windows)}")


    emg_features = []
    for window in windows[:-1]:
        emg_features.append(extract_emg_features(window, False))
    df_emg = pd.concat(emg_features)
    df_emg["ID"] = ID
    Dataframes.append(df_emg)

    cnt += 1
    print(f"Features extracted for {ID}\n")


    del df
    emg_signals = None
    emg_cleaned = None
    del cleaned_df
    windows = None
    emg_features = None
    del df_emg

df_EMG_INC = pd.concat(Dataframes, ignore_index=True)
df_EMG_INC.to_csv(f"/home/vault/empkins/tpD/D02/Students/Yasaman/BioSig_data/Processed Data/EMG/Including_Masseter/EMG_{Window_Time}_Minute.csv", index=False)

# 8 . EMG: Excluding Masseter

In [ ]:
import pandas as pd
import sys
import neurokit2 as nk
import numpy as np
from biosppy.signals import emg
from scipy.signal import welch
import os

In [ ]:
SAMPLING_RATE = 2000
Window_Time = 4 # 10', 5', 4', 3'
WINDOW_SIZE = Window_Time * 60 * 2000

In [ ]:
def apply_windowing(df, window_size=1000):
    # Belongs to Utilities
    """
    Splits a DataFrame into non-overlapping windows of a specified size.

    Parameters:
        df (pd.DataFrame): The input DataFrame to be split into windows.
        window_size (int): The number of rows in each window. Default is 1000.

    Returns:
        list: A list of DataFrames, where each DataFrame represents a window
              containing `window_size` rows. If the number of rows in `df` is
              not perfectly divisible by `window_size`, the last window will
              contain the remaining rows.
    """
    num_windows = len(df) // window_size
    windows = []

    for i in range(num_windows):
        window_start = i * window_size
        window_end = (i + 1) * window_size
        window_data = df.iloc[window_start:window_end]
        windows.append(window_data)

    if len(df) % window_size != 0:
        last_window = df.iloc[num_windows * window_size:]
        windows.append(last_window)

    return windows


def calculate_rmse(emg_signal, sampling_rate):
    ts, filtered, _ = emg.emg(signal=emg_signal, sampling_rate=sampling_rate, show=False)
    return np.sqrt(np.mean(filtered**2))

def calculate_mav(emg_signal, sampling_rate):
    ts, filtered, _ = emg.emg(signal=emg_signal, sampling_rate=sampling_rate, show=False)
    return np.mean(np.abs(filtered))

def calculate_var(emg_signal, sampling_rate):
    ts, filtered, _ = emg.emg(signal=emg_signal, sampling_rate=sampling_rate, show=False)
    return np.var(filtered)

def calculate_energy(emg_signal, sampling_rate):
    ts, filtered, _ = emg.emg(signal=emg_signal, sampling_rate=sampling_rate, show=False)
    return np.sum(filtered**2)


def calculate_mnf_mdf_fr(emg_signal, sampling_rate):
    f, psd = welch(emg_signal, fs=sampling_rate, nperseg=512)

    mnf = np.sum(f * psd) / np.sum(psd)

    cumulative_power = np.cumsum(psd)
    total_power = np.sum(psd)
    mdf_index = np.where(cumulative_power >= total_power / 2)[0][0]
    mdf = f[mdf_index]

    fr_index = np.argmax(psd)
    fr = f[fr_index] / mnf

    return mnf, mdf, fr

def z_score_normalization(signal):
    mean = np.mean(signal)
    std_dev = np.std(signal)
    if std_dev == 0:
        std_dev = 1
    normalized_signal = (signal - mean) / std_dev
    return normalized_signal

def extract_emg_features(dataframe: pd.DataFrame, normalize=False):
    dataframe = dataframe.copy()

    if normalize:
        dataframe['Corrugator'] = z_score_normalization(dataframe['Corrugator'].values)
        dataframe['Oculi'] = z_score_normalization(dataframe['Oculi'].values)
        dataframe['Zygomaticus'] = z_score_normalization(dataframe['Zygomaticus'].values)


    standard_features = dataframe.describe()
    c_mean_value = standard_features.at['mean', 'Corrugator']
    o_mean_value = standard_features.at['mean', 'Oculi']
    z_mean_value = standard_features.at['mean', 'Zygomaticus']

    c_std_value = standard_features.at['std', 'Corrugator']
    o_std_value = standard_features.at['std', 'Oculi']
    z_std_value = standard_features.at['std', 'Zygomaticus']

    c_min_value = standard_features.at['min', 'Corrugator']
    o_min_value = standard_features.at['min', 'Oculi']
    z_min_value = standard_features.at['min', 'Zygomaticus']

    c_max_value = standard_features.at['max', 'Corrugator']
    o_max_value = standard_features.at['max', 'Oculi']
    z_max_value = standard_features.at['max', 'Zygomaticus']

    c_range_value = c_max_value - c_min_value
    o_range_value = o_max_value - o_min_value
    z_range_value = z_max_value - z_min_value



    c_rmse_value = calculate_rmse(dataframe['Corrugator'].values, SAMPLING_RATE)
    o_rmse_value = calculate_rmse(dataframe['Oculi'].values, SAMPLING_RATE)
    z_rmse_value = calculate_rmse(dataframe['Zygomaticus'].values, SAMPLING_RATE)

    c_mav_value = calculate_mav(dataframe['Corrugator'].values, SAMPLING_RATE)
    o_mav_value = calculate_mav(dataframe['Oculi'].values, SAMPLING_RATE)
    z_mav_value = calculate_mav(dataframe['Zygomaticus'].values, SAMPLING_RATE)

    c_var_value = calculate_var(dataframe['Corrugator'].values, SAMPLING_RATE)
    o_var_value = calculate_var(dataframe['Oculi'].values, SAMPLING_RATE)
    z_var_value = calculate_var(dataframe['Zygomaticus'].values, SAMPLING_RATE)

    c_energy_value = calculate_energy(dataframe['Corrugator'].values, SAMPLING_RATE)
    o_energy_value = calculate_energy(dataframe['Oculi'].values, SAMPLING_RATE)
    z_energy_value = calculate_energy(dataframe['Zygomaticus'].values, SAMPLING_RATE)

    c_mnf_value, c_mdf_value, c_fr_value = calculate_mnf_mdf_fr(dataframe['Corrugator'].values, SAMPLING_RATE)
    o_mnf_value, o_mdf_value, o_fr_value = calculate_mnf_mdf_fr(dataframe['Oculi'].values, SAMPLING_RATE)
    z_mnf_value, z_mdf_value, z_fr_value = calculate_mnf_mdf_fr(dataframe['Zygomaticus'].values, SAMPLING_RATE)

    dataframe_final = pd.DataFrame({
        'emg_c_mean': [c_mean_value],
        'emg_o_mean': [o_mean_value],
        'emg_z_mean': [z_mean_value],

        'emg_c_std': [c_std_value],
        'emg_o_std': [o_std_value],
        'emg_z_std': [z_std_value],

        'emg_c_min': [c_min_value],
        'emg_o_min': [o_min_value],
        'emg_z_min': [z_min_value],

        'emg_c_max': [c_max_value],
        'emg_o_max': [o_max_value],
        'emg_z_max': [z_max_value],

        'emg_c_range': [c_range_value],
        'emg_o_range': [o_range_value],
        'emg_z_range': [z_range_value],

        'emg_c_rmse': [c_rmse_value],
        'emg_o_rmse': [o_rmse_value],
        'emg_z_rmse': [z_rmse_value],

        'emg_c_mav': [c_mav_value],
        'emg_o_mav': [o_mav_value],
        'emg_z_mav': [z_mav_value],

        'emg_c_var': [c_var_value],
        'emg_o_var': [o_var_value],
        'emg_z_var': [z_var_value],

        'emg_c_energy': [c_energy_value],
        'emg_o_energy': [o_energy_value],
        'emg_z_energy': [z_energy_value],

        'emg_c_mnf': [c_mnf_value],
        'emg_o_mnf': [o_mnf_value],
        'emg_z_mnf': [z_mnf_value],

        'emg_c_mdf': [c_mdf_value],
        'emg_o_mdf': [o_mdf_value],
        'emg_z_mdf': [z_mdf_value],

        'emg_c_fr': [c_fr_value],
        'emg_o_fr': [o_fr_value],
        'emg_z_fr': [z_fr_value],

    })
    return dataframe_final

In [ ]:
Dataframes = []
cnt = 1

for ID in IDs:
    df = pd.read_csv(f'/home/vault/empkins/tpD/D02/Students/Yasaman/BioSig_data/{ID}/Data.csv')
    print(f"{cnt}. Loaded Signal for {ID}")

    emg_signals = [df['Corrugator'].values, df['Oculi'].values, df['Zygomaticus'].values]
    emg_cleaned = [nk.emg_clean(emg, sampling_rate=SAMPLING_RATE) for emg in emg_signals]
    cleaned_df = pd.DataFrame({
        'Corrugator': emg_cleaned[0],
        'Oculi': emg_cleaned[1],
        'Zygomaticus': emg_cleaned[2]
    })
    print(f"Cleaned Signal for {ID}")


    windows = apply_windowing(cleaned_df, WINDOW_SIZE)
    print(f"Applied Windowing for {ID}")
    print(f"Windows for {ID} is {len(windows)}")


    emg_features = []
    for window in windows[:-1]:
        emg_features.append(extract_emg_features(window, False))
    df_emg = pd.concat(emg_features)
    df_emg["ID"] = ID
    Dataframes.append(df_emg)


    print(f"Features extracted for {ID}\n")
    cnt += 1


    del df
    emg_signals = None
    emg_cleaned = None
    del cleaned_df
    windows = None
    emg_features = None
    del df_emg

df_EMG_EX = pd.concat(Dataframes, ignore_index=True)
df_EMG_EX.to_csv(f"/home/vault/empkins/tpD/D02/Students/Yasaman/BioSig_data/Processed Data/EMG/Excluding_Masseter/EMG_{Window_Time}_Minute.csv", index=False)